In [1]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

In [2]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [3]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [4]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [5]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [6]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [7]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [8]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [9]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 117,
 'tn': 2614,
 'fp': 23,
 'fn': 246,
 'misclassification_rate': 0.08966666666666667,
 'false_positive_rate': 0.008722032612817596,
 'false_negative_rate': 0.6776859504132231}

### Check results on the test set (new data not yet seen by the model)

In [10]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 27,
 'tn': 862,
 'fp': 12,
 'fn': 99,
 'misclassification_rate': 0.111,
 'false_positive_rate': 0.013729977116704805,
 'false_negative_rate': 0.7857142857142857}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

I am semi confident. After playing with it for 15 minutes, I was able to reach misclassification of 0.111. This gives me a good amount of confidence, however, that is still pretty large. That means that 1 in every 10 subjects will be miscalssified as a bot. That really is pretty crappy. Imagine taking in a dataset of 10,000 subjects. That's 1,110 misclassifications! Not something you would be too proud of. 

### What are potential ramifications of false positives from the model?

You would be allowing a bot to carry out tasks you might want to prevent it from. For example, imagine a new sneaker drop, or just random lottery for some item is occuring. It is run through an online website where certain users are picked at random. If a recaptcha test fails 11% of the time, then your chances of winning such a lottery significantly decrease. This used to be, and probably still is, a real issue for sneakers on the Nike SNKRS app.  

### What are potential ramifications of false negatives from the model?

This could mean that someone could be denied service. Maybe someone is trying to login to a healthcare service or social meida. Being labeled a bot and being locked out can be extremely harmful. It could be denying you a necessary service/software. I would be so upset if I couldn't get into one of my accounts if there was some time sensitive action that needed to be performed. Also, imagine playing a video game and being banned because the developers' crappy algorithm somehow labeled you a bot. Now you can't play. I'd be pissed!